### Problem Statement 3A

3A. Implement credit card fraud detection for given credit card fraud detection Kaggle dataset using Autoencoder and build the model by using the following steps:
Dataset: The dataset contains transactions labeled as fraudulent or non-fraudulent, with a severe class imbalance.
Goal: Identify fraudulent transactions by training an autoencoder only on non-fraudulent transactions.
Steps:
- Preprocess the data by scaling features (features are already PCA-transformed, so scale Amount and Time).
- Train an autoencoder on non-fraudulent transactions to learn typical spending patterns.
- Calculate reconstruction errors for transactions and set a threshold to detect fraud.
- Compile the models with Optimizer, Loss, and Evaluation Metrics

In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import StandardScaler

In [ ]:
# Preprocess the data by scaling features
csv_path = "../LP IV Updated Datasets/AutoEncoder/CreditCard/creditcard.csv"

data = pd.read_csv(csv_path)
print("Data shape:", data.shape)
print("Class counts (0=Normal, 1=Fraud):\n", data['Class'].value_counts())

# Scale the 'Amount' column using StandardScaler
scaler = StandardScaler()
data['Amount'] = scaler.fit_transform(data[['Amount']])

# Drop the 'Time' column as V1-V28 are PCA features
X = data.drop(['Time', 'Class'], axis=1).values
y = data['Class'].values

# Filter normal transactions (Class == 0) for training
X_train_normal = X[y == 0]

# Use all data for testing
X_test = X
y_test = y

print("Normal training samples:", X_train_normal.shape[0])
print("Test samples:", X_test.shape[0])

In [ ]:
# Build the Autoencoder model
input_dim = X_train_normal.shape[1]

model = Sequential()
# Encoder
model.add(Dense(16, activation='relu', input_shape=(input_dim,)))
model.add(Dense(8, activation='relu'))  # Bottleneck layer
# Decoder
model.add(Dense(16, activation='relu'))
model.add(Dense(input_dim, activation='linear'))

# Compile the model with Optimizer, Loss, and Evaluation Metrics
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
# Train an autoencoder on non-fraudulent transactions
history = model.fit(
    X_train_normal, X_train_normal,
    epochs=5,
    batch_size=128,
    validation_split=0.1
)

In [ ]:
# Calculate reconstruction errors and set threshold to detect fraud
# Predict reconstructions on test data
predictions = model.predict(X_test)

# Calculate Mean Squared Error (MSE) for each transaction
mse = np.mean(np.square(X_test - predictions), axis=1)

# Set threshold at 95th percentile of normal transactions
threshold = np.percentile(mse[y_test == 0], 95)
print("Fraud detection threshold:", threshold)

# Transactions with error > threshold are classified as Fraud (1)
y_pred = (mse > threshold).astype(int)

# Display detection results
from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

# Plot error distribution
plt.figure(figsize=(8, 4))
plt.hist(mse[y_test == 0], bins=50, alpha=0.6, label='Normal', range=(0, 20), color='blue')
plt.hist(mse[y_test == 1], bins=50, alpha=0.6, label='Fraud', range=(0, 20), color='red')
plt.axvline(threshold, color='black', linestyle='--', label='Threshold')
plt.title('Reconstruction Error for Normal vs Fraud')
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Count')
plt.legend()
plt.show()